In [1]:
## conda env!!!

# 1. Get Data. 
## 1.1 Get Alleles from config

In [2]:
from config import CLASSES_TO_USE, ORGANISMS_TO_USE, PER_ORGANISM_PER_CLASS_ALLELES

#rewrite classes to use to only I class because it's only class I method
METHOD = 'mhcattnnet'
ORGANISMS_TO_USE = ['Human']
CLASSES_TO_USE = ['I', 'II']
ALLELES = [item  for organism_to_use in ORGANISMS_TO_USE 
                 for class_to_use in CLASSES_TO_USE 
                 for item in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]]
ALLELES

['HLA_A_02_01',
 'HLA_B_27_05',
 'HLA_A_03_01',
 'HLA_B_07_02',
 'HLA_DRB1_01_01',
 'HLA_DRB1_04_01',
 'HLA_DRB1_04_05']

## 1.2 Read Data files. 
* add id column
* add mhc column
* filter by length if needed

In [3]:
# importing the predict module
from mhcnames import normalize_allele_name## ONLY FOR LINUX

import pandas as pd
import os
import sys
import numpy as np


In [4]:
# Read datasets
DECOY_KEY = 'x1'

DATA_PATH = "/home/denklewer/Data/"
INPUT_DATA_PATH = DATA_PATH + 'input/'
TARGET_PATH_TO_PREDICTIONS = DATA_PATH + 'predictions/'+ METHOD+'/'
# per_allele_val_df = {
#     allele_name :pd.read_csv("{}df_{}_{}_val_for_classification.csv".format(INPUT_DATA_PATH,allele_name,DECOY_KEY), index_col=0) for allele_name in ALLELES
# }
# per_allele_train_df = {
#     allele_name :pd.read_csv("{}df_{}_{}_train_for_classification.csv".format(INPUT_DATA_PATH,allele_name,DECOY_KEY), index_col=0) for allele_name in ALLELES
# }
# per_allele_test_df = {
#     allele_name :pd.read_csv("{}df_{}_{}_test_for_classification.csv".format(INPUT_DATA_PATH,allele_name,DECOY_KEY), index_col=0) for allele_name in ALLELES
# }
from data_reading import read_genetech_data

per_allele_train_df, per_allele_val_df = read_genetech_data(DATA_PATH, CLASSES_TO_USE, ALLELES)

assert not 0 in [len(per_allele_val_df[allele_name]) for allele_name in ALLELES], 'One of dfs has length of 0, check data reading'


/home/denklewer/Tools/anacoda3/envs/mhcattnnet/lib/python3.6/site-packages/IPython/core/interactiveshell.py:2818: DtypeWarning: Columns (6,7,8,9,14,21,22,23,24) have mixed types.Specify dtype option on import or set low_memory=False.
  if self.run_code(code, result):


In [5]:
list(per_allele_val_df[ALLELES[-1]].columns.values)

['peptide',
 'nFlank',
 'cFlank',
 'allotype',
 'molecule',
 'data_type',
 'el',
 'ic50',
 'ic50_scaled',
 'is_multiallotypic',
 'mhc_dq1_1',
 'mhc_dq1_2',
 'mhc_dq1_3',
 'mhc_dq1_4',
 'mhc_dp1_1',
 'mhc_dp1_2',
 'mhc_dp1_3',
 'mhc_dp1_4',
 'mhc_dr1_1',
 'mhc_dr1_2',
 'mhc_dr3_1',
 'mhc_dr3_2',
 'mhc_dr4_1',
 'mhc_dr4_2',
 'mhc_dr5_1',
 'mhc_dr5_2',
 'gene_ids',
 'transcript_ids',
 'protein_ids',
 'split',
 'analysis_id',
 'ensembl_id',
 'has_overlapping_9mer_train_vs_test',
 'binder']

In [6]:
def defineClass(allele_name):
    for organism_to_use in ORGANISMS_TO_USE:
        for class_to_use in CLASSES_TO_USE:
            if allele_name in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]:
                return class_to_use

In [7]:
def defineAlleleName(allele_name):
    ## HARDCODE
    if allele_name == 'H2_d class II':
        target_allele_name = 'H2_IAd'
    elif allele_name == 'H2_b class II':
        target_allele_name = 'H2_IAb'
    elif allele_name == 'HLA_DRB1_01_01':
        target_allele_name = 'HLA-DRB1*01:01'
    elif allele_name == 'HLA_DRB1_04_01':
        target_allele_name = 'HLA-DRB1*04:01'
    elif allele_name == 'HLA_DRB1_04_05':
        target_allele_name = 'HLA-DRB1*04:05'    
    else:
        target_allele_name = normalize_allele_name(allele_name)
    return target_allele_name

In [8]:
def add_aux_columns(per_allele_df, columns_to_add={'source_sequence_name', 'mhc_allele', 'length'}):
    for allele_name in per_allele_df.keys():
        if 'source_sequence_name' in columns_to_add:
            per_allele_df[allele_name]['source_sequence_name'] =  'seq' + per_allele_df[allele_name].index.astype(str).values
        if 'mhc_allele' in columns_to_add:
            target_allele = defineAlleleName(allele_name)
            per_allele_df[allele_name]['mhc_allele'] = target_allele
        if 'length' in columns_to_add:
            per_allele_df[allele_name]['length'] = per_allele_df[allele_name]['peptide'].str.len()
            
            
def filter_by_length(per_allele_df, length_bound_class_1=15, length_bound_class_2=30):
    if defineClass(allele_name) == 'I':
        per_allele_df[allele_name] = per_allele_df[allele_name][per_allele_val_df[allele_name]['length'] <= 15]
    else:
        per_allele_df[allele_name] = per_allele_df[allele_name][per_allele_val_df[allele_name]['length'] <= 30]
    


            
add_aux_columns(per_allele_val_df)
add_aux_columns(per_allele_train_df)
# add_aux_columns(per_allele_test_df)



In [9]:
[defineAlleleName(allele_name) for allele_name in ALLELES]

['HLA-A*02:01',
 'HLA-B*27:05',
 'HLA-A*03:01',
 'HLA-B*07:02',
 'HLA-DRB1*01:01',
 'HLA-DRB1*04:01',
 'HLA-DRB1*04:05']

In [10]:
per_allele_val_df[ALLELES[0]].head()

,split,el,allele,peptide,n_flank,c_flank,mhc_a_1,mhc_a_2,mhc_b_1,mhc_b_2,...,n_studies,Sample_ids,sample_names,Study_ids,study_names,negative_ensembl_id,binder,source_sequence_name,mhc_allele,length
52052,test,1,HLA-A*02:01,AGIALNDNFVK,DTHSSIFDAK,LISWYDNEYG,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,1.0,1887,iedb_1034377_EL_A0201,5,iedb_consolidated,NaN,1,seq52052,HLA-A*02:01,11
52053,test,1,HLA-A*02:01,AGLYRGLCAVRSR,GGAGPAVLQV,ALGLGLVSPA,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,1.0,1320,iedb_1029626_EL_A0201,5,iedb_consolidated,NaN,1,seq52053,HLA-A*02:01,13
52054,test,1,HLA-A*02:01,AIDEKVQAV,APDIDKLKMA,HNLKMFWQST,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,1.0,1887,iedb_1034377_EL_A0201,5,iedb_consolidated,NaN,1,seq52054,HLA-A*02:01,9
52055,test,1,HLA-A*02:01,AIFSAIIFL,VRHCFFTVIF,SFEAVSSLSK,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,1.0,677,Sarkizova_A0201,21,sarkizova_mhcI_2020,NaN,1,seq52055,HLA-A*02:01,9
52056,test,1,HLA-A*02:01,AIIGGTFTV,PLYRFITTIC,AGILDSCIFT,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,1.0,"1356,1887","iedb_1019570_EL_A0201,iedb_1034377_EL_A0201",5,iedb_consolidated,NaN,1,seq52056,HLA-A*02:01,9


# 2. Working with predictor

In [11]:
PREDICTOR_HOME = os.getenv('MHCATTNNET_HOME')
sys.path.append(PREDICTOR_HOME)

from src.alleles_I import c1_map
from src.alleles_II import c2_map
PREDICTOR_HOME

'/home/denklewer/Projects/MHCAttnNet'

## 2.1 Preprocess data
* generate tran-test-val (peptide,mhc_acids, bind, mhc_allele)

In [12]:
def add_mhc_sequence_column(per_allele_df):
    for allele_name, df in per_allele_df.items():
        if defineClass(allele_name) == 'I':
            df['mhc_amino_acid'] = df.mhc_allele.apply(lambda x: c1_map.get(x, x))
        else:
            df['mhc_amino_acid'] =  df.mhc_allele.apply(lambda x: c2_map.get(x, x))

            
def rename_column(per_allele_df, initial_name, target_name):
    for allele_name, df in per_allele_df.items():
        df.rename(columns={initial_name: target_name}, inplace=True)
            
add_mhc_sequence_column(per_allele_train_df)
# add_mhc_sequence_column(per_allele_test_df)
add_mhc_sequence_column(per_allele_val_df)


rename_column(per_allele_train_df, 'binder', 'bind')
# rename_column(per_allele_test_df,'binder', 'bind')
rename_column(per_allele_val_df, 'binder', 'bind')


# Join dfs by class
def get_per_class_dfs(per_allele_dfs):
    per_class_df = dict()
    for class_to_use in CLASSES_TO_USE:
        class_dfs_list = [per_allele_dfs[allele_name]  for allele_name in per_allele_dfs.keys() if defineClass(allele_name) == class_to_use ]
        class_result_df = pd.concat(class_dfs_list)
        per_class_df[class_to_use] = class_result_df
    return per_class_df

per_class_train_df = get_per_class_dfs(per_allele_train_df)
# per_class_test_df  = get_per_class_dfs(per_allele_test_df)
per_class_val_df  = get_per_class_dfs(per_allele_val_df)


In [13]:
per_class_val_df['II'][[
    #'mhc_amino_acid','mhc_allele',
    'mhc_dq1_1',
        'allotype',
       'mhc_dq1_2', 'mhc_dq1_3', 'mhc_dq1_4', 'mhc_dp1_1', 'mhc_dp1_2',
       'mhc_dp1_3', 'mhc_dp1_4', 'mhc_dr1_1', 'mhc_dr1_2', 'mhc_dr3_1',
       'mhc_dr3_2', 'mhc_dr4_1', 'mhc_dr4_2', 'mhc_dr5_1', 'mhc_dr5_2']].head().iloc[0]['mhc_dr1_1']

'QEFFIASGAAVDAIM_WLFLECYDLQRATYHVGFT'

In [14]:
per_class_val_df['II'][['mhc_amino_acid','mhc_allele','mhc_dq1_1',
       'mhc_dq1_2', 'mhc_dq1_3', 'mhc_dq1_4', 'mhc_dp1_1', 'mhc_dp1_2',
       'mhc_dp1_3', 'mhc_dp1_4', 'mhc_dr1_1', 'mhc_dr1_2', 'mhc_dr3_1',
       'mhc_dr3_2', 'mhc_dr4_1', 'mhc_dr4_2', 'mhc_dr5_1', 'mhc_dr5_2']].head().iloc[0]['mhc_amino_acid']

'RFLWQLKFECHFFNGTERVRLLERCIYNQEESVRFDSDVGEYRAVTELGRPDAEYWNSQKDLLEQRRAAVDTYCRHNYGVGESFTVQRR'

In [15]:
per_class_val_df['II'][['mhc_amino_acid','mhc_allele','mhc_dq1_1',
       'mhc_dq1_2', 'mhc_dq1_3', 'mhc_dq1_4', 'mhc_dp1_1', 'mhc_dp1_2',
       'mhc_dp1_3', 'mhc_dp1_4', 'mhc_dr1_1', 'mhc_dr1_2', 'mhc_dr3_1',
       'mhc_dr3_2', 'mhc_dr4_1', 'mhc_dr4_2', 'mhc_dr5_1', 'mhc_dr5_2']].tail().iloc[0]['mhc_amino_acid']

'RFLEQVKHECHFFNGTERVRFLDRYFYHQEEYVRFDSDVGEYRAVTELGRPSAEYWNSQKDLLEQRRAAVDTYCRHNYGVGESFTVQRRVYPEVTVYPAKTQPLQHHNLLVCSVNGFYPGSIEVRWFRNGQEEKTGVVSTGLIQNGDWTFQTLVMLETVPRSGEVYTCQVEHPSLTSPLTVEWRARSESAQSKMLSGVGGFVLGLLFLGAGLFIYFRNQKGHSGLQPTGFLS'

## 2.2 Save Preprocessed csv to mhcattnet data dir (for training)

In [16]:
PATH_TO_DATA = f'{PREDICTOR_HOME}/data'

def reorder_columns(per_class_df):
    for class_to_use,df in per_class_df.items():
        df = df[['peptide', 'mhc_amino_acid', 'bind', 'mhc_allele', 'source_sequence_name']]
        per_class_df[class_to_use] = df
        
reorder_columns(per_class_train_df)
# reorder_columns(per_class_test_df)
reorder_columns(per_class_val_df)
for class_to_use in CLASSES_TO_USE:
    
    per_class_train_df[class_to_use].to_csv(f"{PATH_TO_DATA}/class{class_to_use}/train.csv",index=False)
#     per_class_test_df[class_to_use].to_csv(f"{PATH_TO_DATA}/class{class_to_use}/test.csv",index=False)
    per_class_val_df[class_to_use].to_csv(f"{PATH_TO_DATA}/class{class_to_use}/val.csv",index=False)



## 2.3 Train(If needed)

In [17]:
#  use virtual machine  in the folder $MHCATTNNET_HOME
# python src/train.py -c 1
#  python src/train.py -c 2

## 2.4 Evaluate trained models

In [17]:
from tqdm import tqdm

In [18]:
from src import config
import torch

/home/denklewer/Tools/anacoda3/envs/mhcattnnet/lib/python3.6/site-packages/torch/cuda/__init__.py:52: UserWarning: CUDA initialization: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx (Triggered internally at  /pytorch/c10/cuda/CUDAFunctions.cpp:100.)
  return torch._C._cuda_getDeviceCount() > 0


In [19]:
from src.model import MHCAttnNet
from src.data_loader import DataLoader

In [20]:
def run_validation_for_class(class_to_use):
    config.read_all_columns = True
    if class_to_use == 'I':
        config.model_name = config.mhc_1_model_name
        config.base_path = config.mhc_1_base_path
    else:
        config.model_name = config.mhc_2_model_name
        config.base_path = config.mhc_2_base_path
    device = config.device
    data_loader_obj  = DataLoader(config)
    train_loader, val_loader, test_loader, peptide_embedding, mhc_embedding = data_loader_obj.get_dataset(device, shuffle=False)
    ## Read model
    model = MHCAttnNet(peptide_embedding, mhc_embedding, config)
    model.load_state_dict(torch.load(config.model_name))
    model.to(device)
    y_actual_val = []
    y_pred_val = []
    seq_id = []
    mhc_allele = []
    for row in tqdm(val_loader):
        result = model.forward(row.peptide,row.mhc_amino_acid)
        y_actual = row.bind
        y_actual_val += list(y_actual.cpu().data.numpy())
        result_idx = torch.max(result, dim=1)[1]
        y_pred_val += list(result_idx.cpu().data.numpy())
        seq_id += list(row.source_sequence_name)
        mhc_allele += list(row.mhc_allele)
    return pd.DataFrame({"source_sequence_name":seq_id, "binder":y_actual_val, "score": y_pred_val, "mhc_allele":mhc_allele})


In [22]:
%%time
for class_to_use in CLASSES_TO_USE:
     per_class_val_df[class_to_use] =  run_validation_for_class(class_to_use)

/home/denklewer/Tools/anacoda3/envs/mhcattnnet/lib/python3.6/site-packages/torchtext/data/field.py:150: UserWarning: Field class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.
  warnings.warn('{} class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.'.format(self.__class__.__name__), UserWarning)
/home/denklewer/Tools/anacoda3/envs/mhcattnnet/lib/python3.6/site-packages/torchtext/data/field.py:36: UserWarning: RawField class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.
  warnings.warn('{} class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.'.format(self.__class__.__name__), UserWarning)
/home/denklewer/Tools/anacoda3/envs/mhcattnnet/lib/python3.6/site-packages/torchtext/data/example.py:68: UserWarning: Exam

Peptide embedding dimension torch.Size([22, 100])
MHC Amino Acid embedding dimension torch.Size([136, 100])


/home/denklewer/Tools/anacoda3/envs/mhcattnnet/lib/python3.6/site-packages/torchtext/data/batch.py:23: UserWarning: Batch class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.
  warnings.warn('{} class will be retired soon and moved to torchtext.legacy. Please see the most recent release notes for further information.'.format(self.__class__.__name__), UserWarning)
  0%|          | 0/881 [00:00<?, ?it/s]

Peptide embedding dimension torch.Size([22, 100])
MHC Amino Acid embedding dimension torch.Size([88, 100])


100%|██████████| 881/881 [02:43<00:00,  5.40it/s]


CPU times: user 1h 4min 9s, sys: 41.8 s, total: 1h 4min 51s
Wall time: 35min 16s


## View results

In [23]:
per_class_val_df[CLASSES_TO_USE[1]]

,source_sequence_name,binder,score,mhc_allele
0,seq59239,1,1,HLA-DRB1*01:01
1,seq59694,1,0,HLA-DRB1*01:01
2,seq59737,1,1,HLA-DRB1*01:01
3,seq60038,1,0,HLA-DRB1*01:01
4,seq60055,1,0,HLA-DRB1*01:01
...,...,...,...,...
28165,seq87235,0,0,HLA-DRB1*04:05
28166,seq87238,0,0,HLA-DRB1*04:05
28167,seq87259,0,0,HLA-DRB1*04:05
28168,seq87262,0,0,HLA-DRB1*04:05


## Split by allele

In [24]:
per_allele_predictions = dict()
for allele_name in ALLELES:
    target_allele_name = defineAlleleName(allele_name)
    class_of_allele = defineClass(allele_name)
    
    df = per_class_val_df[class_of_allele]
    df = df[df['mhc_allele'] == target_allele_name]
    per_allele_predictions[allele_name] = per_allele_val_df[allele_name].merge( df[['source_sequence_name', 'score']] , on=['source_sequence_name'])
    per_allele_predictions[allele_name] = per_allele_predictions[allele_name][['source_sequence_name','peptide', 'mhc_allele', 'score', 'bind']]

rename_column(per_allele_predictions, 'bind', 'binder')    
    

#### write to files

In [25]:
# for decoy_key in DECOY_PREFIXES:
for allele_name, total_df in per_allele_predictions.items():
    total_df.to_csv("{}{}_predictions_{}".format(TARGET_PATH_TO_PREDICTIONS,METHOD,allele_name))

In [26]:
total_df

,source_sequence_name,peptide,mhc_allele,score,binder
0,seq59198,ADAWIQAIKSLAEKQN,HLA-DRB1*04:05,1,1
1,seq59199,ADAWIQAIKSLAEKQNL,HLA-DRB1*04:05,1,1
2,seq59207,AEDNADTLALVFEAPNQEK,HLA-DRB1*04:05,0,1
3,seq59225,AERGYLYFTNMQDRAA,HLA-DRB1*04:05,1,1
4,seq59226,AERGYLYFTNMQDRAAKIE,HLA-DRB1*04:05,1,1
...,...,...,...,...,...
1595,seq87275,TDMNRHHYS,HLA-DRB1*04:05,0,0
1596,seq87276,AKAKAPKPKAAKPKSGKPKV,HLA-DRB1*04:05,1,0
1597,seq87277,TLLTAISMSAIATNGVVPAGGSYFMI,HLA-DRB1*04:05,0,0
1598,seq87278,QPTLVAVGKSFTIECRVP,HLA-DRB1*04:05,0,0
